### Selecting the optimal pipeline for answer generation

This notebook evaluates the following pipelines to determine the most effective approach for generating answers:
1. Basic Generation: Context → Plan generation → Answer generation
2. Theoretical Generation: Context → Plan generation with applicable theories → Answer generation
3. Context Filtering Only: Context → Filtered context generation → Answer generation (no planning or theoretical framing)
4. Direct Generation: Context → Answer generation (no preprocessing stage)

In [ ]:
#imports

import os
import sys
import tqdm
import pandas as pd
import logging
import warnings

#some important stuff setup

#here can be mistakes bc I've moved this file from NIR567 to NIR567/notebooks; i've changes paths, but i'm not sure in their correctness
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(project_root)
sys.path.insert(0, project_root)

logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("faiss").setLevel(logging.WARNING)
warnings.filterwarnings("ignore")

#this nir imports

from nir.llm.manager import ModelManager
from nir.llm.providers import ModelConfig

from nir.tests.test_datasets import TEST_DATA_TEXT1
from nir.tests.evaluator import analyze_generation

from nir.graph.graph_storages.networkx_graph import NetworkXGraph
from nir.core.context_retriever import form_context_with_llm
from nir.core.answers_generator import generate_plan
from nir.core.answers_generator import filter_context
from nir.core.answers_generator import generate_answer_based_on_plan
from nir.core.answers_generator import generate_answer_based_on_context

In [ ]:
#models setup

manager = ModelManager()

instruct_model_config = ModelConfig(model_name="hf.co/VlSav/Vikhr-Nemo-12B-Instruct-R-21-09-24-Q4_K_M-GGUF:latest", temperature=0.0)
instruct_llm = manager.create_chat_model(name="test_model", option="ollama", config=instruct_model_config)

answer_model_config = ModelConfig(model_name="llama3.2:latest", temperature=0.7)
answer_llm = manager.create_chat_model(name="test_model", option="ollama", config=instruct_model_config)

embeddings_model = manager.create_embedding_model(name="embeddings", option="hf_local", model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
#data setup

test_data = TEST_DATA_TEXT1
graph = NetworkXGraph()
graph.load(filepath="assets/graphs/elden_ring_lore.json")

**Testing basic generation**

In [ ]:
all_metrics = []
generated = []

for task in tqdm.tqdm(test_data["tasks"], desc=f"Testing basic generation"):
    query = task["query"]
    reference = task["reference"]
    category = task.get("category", "default")

    context = form_context_with_llm(query=query, graph=graph, llm=instruct_llm, embedding_model=embeddings_model, language="en", add_history=False, max_tokens=1024)
    answer_plan = generate_plan(query=query, context=context, llm=answer_llm, include_theory=False, language="en")
    answer_final = generate_answer_based_on_plan(query=query, plan=answer_plan, context=context, llm=answer_llm, language="en")
    generated.append({ "category": category,"generated_text": answer_final })
    
    metrics = analyze_generation(
        generated_text=answer_final,
        context=context,
        lore_summary=test_data.get("text_summary", ""),
        reference_text=reference,
        query=query,
        category=category,
        evaluation_llm=answer_llm,
        embedding_model=embeddings_model,
        language="en"
    )

    all_metrics.append(metrics)

Testing basic generation: 100%|██████████| 5/5 [38:57<00:00, 467.52s/it]


In [ ]:
if not all_metrics:
    display(pd.DataFrame({"status": ["No data for analysis"]}))
else:
    df = pd.DataFrame(all_metrics)
    num_cols = df.select_dtypes(include="number").columns.tolist()
    if "category" in df.columns and len(df) > 0:
        cat_df = df.groupby("category")[num_cols].mean().reset_index()
    else:
        cat_df = df[num_cols].mean().to_frame().T
        cat_df["category"] = "default"
    overall = {col: df[col].mean() for col in num_cols}
    overall["category"] = "OVERALL"
    overall_df = pd.DataFrame([overall])

    final_df = pd.concat([cat_df, overall_df], ignore_index=True)
    cols_order = ["category"] + sorted([c for c in final_df.columns if c != "category"])
    final_df = final_df[cols_order]

    display(final_df.style.format(precision=4))

if generated:
    gen_df = pd.DataFrame(generated)
    styled_gen = gen_df.style.set_properties(
        subset=["generated_text"],
        **{
            "white-space": "normal",
            "overflow-wrap": "break-word", 
            "vertical-align": "top", 
            "text-align": "left",
            "max-width": "1200px", 
            "padding": "8px" 
        }
    ).hide(axis="index") 
    display(styled_gen)
else:
    display(pd.DataFrame({"status": ["No generated answers"]}))

,category,bert_score_reference,bert_score_source,distinct_2,repetition_2,world_consistency
0,character description,0.7885,0.8055,0.8600,0.0769,0.9570
1,dialogue,0.8005,0.7952,0.9024,0.0621,0.8570
2,item description,0.7959,0.8080,0.8804,0.0677,0.9570
3,location description,0.7902,0.7946,0.8398,0.0664,0.9500
4,quest,0.7881,0.8027,0.8634,0.0829,0.9570
5,OVERALL,0.7926,0.8012,0.8692,0.0712,0.9356


category,generated_text
quest,"In the heart of the Lands Between, where the echoes of ancient wars still resonate through the shattered remnants of the Elden Ring, a small village clings to life. It is here that the Tarnished finds themselves, drawn by whispers of a hidden truth that could unravel the very fabric of the Golden Order. **The Hidden Faction** Upon arrival, the Tarnished encounters a group known as the Veiled Seekers, a minor faction with eyes that hold secrets deeper than any crypt. They speak of an artifact, long lost to time and war, which they believe holds the key to understanding Marika's true intentions during the Shattering. **The Quest for Knowledge** Tasked with retrieving this ancient relic from the ruins of an ancient library, the Tarnished ventures forth, battling through remnants of Fire Giants and other creatures that once fought against Queen Marika. The journey is fraught with peril, but the promise of knowledge drives them on. **The Artifact's Power** Upon obtaining the artifact, a device capable of channeling the very essence of the Greater Will, the Tarnished learns that it can be wielded to either fortify or dismantle the Golden Order. The Veiled Seekers reveal their true allegiance and the potential for chaos that the Tarnished now holds in their hands. **The Moral Crossroads** Confronted with a choice that could alter the fate of the Lands Between, the Tarnished must decide where their loyalties lie. Do they side with the faction, defying the Golden Order and risking the wrath of Radagon? Or do they report back to a representative of the order, potentially strengthening its iron grip on the realm? **The Consequences Unfold** Each decision leads down a path that will forever change the Tarnished's standing in the world. Allies become enemies, and enemies offer unexpected aid as the consequences of their choice ripple out like shockwaves. **Epilogue: The Truth Revealed** As the quest reaches its climax, the Tarnished faces a final confrontation or revelation that ties up the loose ends of their journey. They learn that Marika's actions were not solely those of a tyrant but a mother torn between her children and the will of the Greater Will. The artifact's true purpose is revealed, offering a glimpse into the truth behind the Shattering. In the end, the Tarnished stands as a beacon of change or a pillar of the old order, their choice etched into the very soul of the Lands Between."
dialogue,"In the heart of the Lands Between, where whispers of ancient power linger in the air, you find yourself standing before an ethereal figure. Its form shimmers like the first light of dawn, and its eyes hold the depth of the cosmos. Spirit: ""Greetings, Tarnished, for I have been expecting your return. The Erdtree's roots run deep with the will of Queen Marika, and it is through them that I speak to you now."" Player: [a nod, acknowledging the presence] Spirit: ""I am a guardian of the Golden Order, a servant to the Greater Will's eternal design. In these trying times, as the Elden Ring lies shattered, we seek allies who would stand with us in its restoration."" Player: [curiosity piqued, you lean in slightly] Spirit: ""A pact I offer, one that binds your fate to the power of the Erdtree and the mission of the Golden Order. Together, we shall uphold the will of the Greater Will and bring order to chaos."" Player: [hesitation, yet a spark of ambition ignites within you] Spirit: ""Know this, Tarnished, with the gift of power comes the burden of knowledge. Secrets long buried may rise to challenge your understanding of the world and your place in it."" Player: [a deep breath, considering the weight of the offer] Spirit: ""Will you accept this pact? Will you become a beacon of hope in the Lands Between, or shall you forge your own path?"" [The spirit extends a hand, palm open, a soft glow emanating from its center. A choice lies before you—a choice that will shape not only your journey but the very destiny of the

**Testing theoretical generation**

In [ ]:
all_metrics = []
generated = []

for task in tqdm.tqdm(test_data["tasks"], desc=f"Testing theoretical generation"):
    query = task["query"]
    reference = task["reference"]
    category = task.get("category", "default")

    context = form_context_with_llm(query=query, graph=graph, llm=instruct_llm, embedding_model=embeddings_model, language="en", add_history=False, max_tokens=1024)
    answer_plan = generate_plan(query=query, context=context, llm=answer_llm, include_theory=True, language="en")
    answer_final = generate_answer_based_on_plan(query=query, plan=answer_plan, context=context, llm=answer_llm, language="en")
    generated.append({ "category": category,"generated_text": answer_final })
    
    metrics = analyze_generation(
        generated_text=answer_final,
        context=context,
        lore_summary=test_data.get("text_summary", ""),
        reference_text=reference,
        query=query,
        category=category,
        evaluation_llm=answer_llm,
        embedding_model=embeddings_model,
        language="en"
    )

    all_metrics.append(metrics)

Testing theoretical generation: 100%|██████████| 5/5 [39:42<00:00, 476.50s/it]


In [ ]:
if not all_metrics:
    display(pd.DataFrame({"status": ["No data for analysis"]}))
else:
    df = pd.DataFrame(all_metrics)
    num_cols = df.select_dtypes(include="number").columns.tolist()
    if "category" in df.columns and len(df) > 0:
        cat_df = df.groupby("category")[num_cols].mean().reset_index()
    else:
        cat_df = df[num_cols].mean().to_frame().T
        cat_df["category"] = "default"
    overall = {col: df[col].mean() for col in num_cols}
    overall["category"] = "OVERALL"
    overall_df = pd.DataFrame([overall])

    final_df = pd.concat([cat_df, overall_df], ignore_index=True)
    cols_order = ["category"] + sorted([c for c in final_df.columns if c != "category"])
    final_df = final_df[cols_order]

    display(final_df.style.format(precision=4))

if generated:
    gen_df = pd.DataFrame(generated)
    styled_gen = gen_df.style.set_properties(
        subset=["generated_text"],
        **{
            "white-space": "normal",
            "overflow-wrap": "break-word", 
            "vertical-align": "top", 
            "text-align": "left",
            "max-width": "1200px", 
            "padding": "8px" 
        }
    ).hide(axis="index") 
    display(styled_gen)
else:
    display(pd.DataFrame({"status": ["No generated answers"]}))

,category,bert_score_reference,bert_score_source,distinct_2,repetition_2,world_consistency
0,character description,0.7906,0.8061,0.8571,0.0826,0.9500
1,dialogue,0.7971,0.7911,0.8564,0.0836,0.9570
2,item description,0.7917,0.8033,0.8434,0.0805,0.9570
3,location description,0.7926,0.7937,0.8351,0.0651,0.9570
4,quest,0.7809,0.8035,0.8531,0.0619,0.9570
5,OVERALL,0.7906,0.7995,0.8490,0.0747,0.9556


category,generated_text
quest,"In the shadowed corners of the Lands Between, where whispers of defiance echo against the iron grip of the Golden Order, a secret society known as the Veiled Seekers had long harbored dreams of a world free from the tyranny of the Greater Will. It was here that the Tarnished's journey would intertwine with the fate of a village plagued by disappearances, leading to revelations that would shake the very foundations of their loyalty. ### Introduction The Tarnished arrived in the quaint village of Eldenholt, where the air was thick with unease. The once vibrant community had been stricken by a mysterious blight, its people vanishing without trace under the cloak of night. A frail villager, eyes wide with desperation, approached the Tarnished with a plea for help, hinting at a connection to the Shattering that had rent the world asunder. ### Inciting Incident Guided by the villager's words, the Tarnished unearthed an ancient artifact hidden beneath the roots of the decaying Erdtree. As they touched its surface, visions flooded their mind—revelations of the Golden Order's past actions, a tapestry woven with threads of deceit and manipulation. ### Rising Action The quest led the Tarnished through treacherous paths, encountering both the zealous followers of the Golden Order and those who dared to defy its edicts. Each step brought moral dilemmas that forced them to choose between supporting or undermining the Order's influence. The choices were not without consequence; each decision left a mark on the Tarnished's soul. ### Climax In the heart of a crumbling fortress, the Tarnished confronted a high-ranking member of the Golden Order. As blades clashed and secrets spilled, they learned of a shocking truth: Queen Marika had not acted alone in her quest to shatter the Elden Ring. A secret alliance with Hoarah Loux (Godfrey) had been forged, one that sought to reshape the world according to their own designs, hidden from the eyes of the Greater Will. ### Falling Action Armed with this knowledge, the Tarnished's path diverged. They could use the artifact to expose the Order's lies and rally the Veiled Seekers to their cause, or they could remain silent, allowing the village to continue its life in ignorance. The fate of Eldenholt hung in the balance, a pendulum swaying between hope and despair. ### Resolution The Tarnished's final choice would determine the outcome of their quest. Would they stand with the Golden Order, upholding the will of the Greater Will, or would they defy it, forging a new path for themselves and those who followed? The decision was made in the heat of battle, as the Tarnished faced down a foe whose loyalty was as fickle as the winds. ### Epilogue As the dust settled on Eldenholt, the consequences of the Tarnished's actions became clear. The village flourished or withered based on their choice, and the Veiled Seekers either gained a new champion or lost another soul to the Order's might. The Tarnished's journey was far from over; the Lands Between were vast, and the secrets they held ran deep. In this side quest, the Tarnished grappled with the moral ambiguity of loyalty and defiance, uncovering hidden truths that challenged their understanding of the world and their place within it. Their choices would echo through the ages, a testament to the power of individual will in the face of divine decree."
dialogue,"In the heart of the ancient forest, where the whispers of the Erdtree intertwine with the very essence of the Lands Between, you find yourself standing before an ethereal figure. Its form shifts like mist, one moment a guardian of old, the next a reflection of your own destiny. **Spirit:** ""Greetings, Tarnished. I am but a humble servant of the Greater Will, bound to the Erdtree's eternal vigil. You have been called here for a purpose greater than any you could fathom."" **Player:** *nods, listening intently* **Spirit:** ""The pact I offer is not one of mere words, but a bond fo

**Testing context filtering only generation**

In [ ]:
all_metrics = []
generated = []

for task in tqdm.tqdm(test_data["tasks"], desc=f"Testing filtering only generation"):
    query = task["query"]
    reference = task["reference"]
    category = task.get("category", "default")

    context = form_context_with_llm(query=query, graph=graph, llm=instruct_llm, embedding_model=embeddings_model, language="en", add_history=False, max_tokens=1024)
    filtered_context = filter_context(query=query, context=context, llm=answer_llm, language="en")
    answer_final = generate_answer_based_on_context(query=query, context=filtered_context, llm=answer_llm, language="en")
    generated.append({ "category": category,"generated_text": answer_final })
    
    metrics = analyze_generation(
        generated_text=answer_final,
        context=context,
        lore_summary=test_data.get("text_summary", ""),
        reference_text=reference,
        query=query,
        category=category,
        evaluation_llm=answer_llm,
        embedding_model=embeddings_model,
        language="en"
    )

    all_metrics.append(metrics)

Testing filtering only generation: 100%|██████████| 5/5 [32:37<00:00, 391.48s/it]


In [ ]:
if not all_metrics:
    display(pd.DataFrame({"status": ["No data for analysis"]}))
else:
    df = pd.DataFrame(all_metrics)
    num_cols = df.select_dtypes(include="number").columns.tolist()
    if "category" in df.columns and len(df) > 0:
        cat_df = df.groupby("category")[num_cols].mean().reset_index()
    else:
        cat_df = df[num_cols].mean().to_frame().T
        cat_df["category"] = "default"
    overall = {col: df[col].mean() for col in num_cols}
    overall["category"] = "OVERALL"
    overall_df = pd.DataFrame([overall])

    final_df = pd.concat([cat_df, overall_df], ignore_index=True)
    cols_order = ["category"] + sorted([c for c in final_df.columns if c != "category"])
    final_df = final_df[cols_order]

    display(final_df.style.format(precision=4))

if generated:
    gen_df = pd.DataFrame(generated)
    styled_gen = gen_df.style.set_properties(
        subset=["generated_text"],
        **{
            "white-space": "normal",
            "overflow-wrap": "break-word", 
            "vertical-align": "top", 
            "text-align": "left",
            "max-width": "1200px", 
            "padding": "8px" 
        }
    ).hide(axis="index") 
    display(styled_gen)
else:
    display(pd.DataFrame({"status": ["No generated answers"]}))

,category,bert_score_reference,bert_score_source,distinct_2,repetition_2,world_consistency
0,character description,0.7908,0.7990,0.9439,0.0495,0.9570
1,dialogue,0.8109,0.8071,0.9107,0.0491,0.9570
2,item description,0.8078,0.8033,0.8638,0.0778,0.9500
3,location description,0.8018,0.8008,0.8190,0.0825,0.9500
4,quest,0.8309,0.8364,0.8171,0.0694,0.9570
5,OVERALL,0.8085,0.8093,0.8709,0.0657,0.9542


category,generated_text
quest,"**Quest Title:** Echoes of Defiance **Objective:** Uncover the truth behind a mysterious artifact that holds the power to challenge the Golden Order. **Quest Giver:** A lone scholar named Elara, who has been studying ancient texts and believes she has found something that could change the fate of the Lands Between. **Steps:** 1. **The Scholar's Summons** - The Tarnished receives a cryptic message from Elara, requesting their aid in deciphering an ancient artifact. - Travel to Elara's hidden sanctuary on the outskirts of a war-torn village. 2. **Deciphering the Past** - Assist Elara in translating the artifact's inscriptions, which hint at a secret history of Marika and her relationship with the Greater Will. - Discover that the artifact was created by Hoarah Loux (Godfrey) before he aligned himself with Marika, suggesting a hidden truth about his loyalties. 3. **The Village's Plight** - The village is under siege from a faction seeking to harness the power of the artifact for their own gain. - Protect the villagers and learn that they have been suffering due to the Golden Order's strict enforcement of the Greater Will's will, which has led to harsh living conditions. 4. **The Artifact's Power** - Realize that the artifact can potentially restore or alter the Elden Ring, affecting the influence of the Greater Will. - Elara reveals her belief that the artifact could be used to free the people from the oppressive grasp of the Golden Order. 5. **A Choice of Loyalty** - The Tarnished must decide whether to: - Return the artifact to the Golden Order, upholding their commitment to the Greater Will's will. - Side with Elara and use the artifact to challenge the Golden Order, risking defiance against Marika and the Greater Will. 6. **Resolution** - If the Tarnished chooses loyalty to the Golden Order: - Deliver the artifact to a high-ranking official of the Golden Order in exchange for clemency for the village. - The village is saved from immediate danger, but the people's spirits remain crushed under the weight of the Order's rule. - If the Tarnished defies the Golden Order: - Activate the artifact with Elara's help, causing a ripple effect that disrupts the stability of the Lands Between. - Face the consequences of this action, which could lead to new quests or further exploration of the game's lore regarding the true nature of the Greater Will and Marika. **Reward:** - Depending on the choice made, the Tarnished gains either the favor of the Golden Order or the gratitude of Elara and the villagers. - Unlock a new ability or item that reflects the Tarnished's alignment with their chosen path."
dialogue,"**Spirit:** Wanderer of the Lands Between, I have awaited one such as you. The path ahead is fraught with peril, but together we may unravel the mysteries that bind our world. **Player:** Who are you? **Spirit:** I am a remnant of the Outer Gods' influence, a spirit bound to the will of the lands themselves. I offer you a pact, one that will grant you knowledge and power beyond your wildest dreams. **Player:** What is this pact? **Spirit:** It is an ancient covenant, sealed with the blood of the gods and the essence of the Erdtree. Through it, we shall work in tandem to uncover the truths hidden from the eyes of mortals. **Player:** And what role do I play in all this? **Spirit:** You are the key, Tarnished. Your journey will reveal the secrets of the Greater Will, the true nature of Queen Marika and her involvement with the Golden Order. The Elden Ring's fate rests upon your shoulders, and through our pact, you shall wield the power to influence the very fabric of the Lands Between. **Player:** I accept this pact. What must I do? **Spirit:** Trust in your instincts, seek out the echoes of the past, and let the whispers of the Outer Gods guide your steps. Together, we shall challenge the very foundations of the world as you know it."
character description,"Character Name: Eolande, the Wolfsblood Renegade

**Testing direct generation**

In [ ]:
all_metrics = []
generated = []

for task in tqdm.tqdm(test_data["tasks"], desc=f"Testing direct generation"):
    query = task["query"]
    reference = task["reference"]
    category = task.get("category", "default")

    context = form_context_with_llm(query=query, graph=graph, llm=instruct_llm, embedding_model=embeddings_model, language="en", add_history=False, max_tokens=1024)
    answer_final = generate_answer_based_on_context(query=query, context=context, llm=answer_llm, language="en")
    generated.append({ "category": category,"generated_text": answer_final })
    
    metrics = analyze_generation(
        generated_text=answer_final,
        context=context,
        lore_summary=test_data.get("text_summary", ""),
        reference_text=reference,
        query=query,
        category=category,
        evaluation_llm=answer_llm,
        embedding_model=embeddings_model,
        language="en"
    )

    all_metrics.append(metrics)

Testing direct generation: 100%|██████████| 5/5 [23:13<00:00, 278.66s/it]


In [ ]:
if not all_metrics:
    display(pd.DataFrame({"status": ["No data for analysis"]}))
else:
    df = pd.DataFrame(all_metrics)
    num_cols = df.select_dtypes(include="number").columns.tolist()
    if "category" in df.columns and len(df) > 0:
        cat_df = df.groupby("category")[num_cols].mean().reset_index()
    else:
        cat_df = df[num_cols].mean().to_frame().T
        cat_df["category"] = "default"
    overall = {col: df[col].mean() for col in num_cols}
    overall["category"] = "OVERALL"
    overall_df = pd.DataFrame([overall])

    final_df = pd.concat([cat_df, overall_df], ignore_index=True)
    cols_order = ["category"] + sorted([c for c in final_df.columns if c != "category"])
    final_df = final_df[cols_order]

    display(final_df.style.format(precision=4))

if generated:
    gen_df = pd.DataFrame(generated)
    styled_gen = gen_df.style.set_properties(
        subset=["generated_text"],
        **{
            "white-space": "normal",
            "overflow-wrap": "break-word", 
            "vertical-align": "top", 
            "text-align": "left",
            "max-width": "1200px", 
            "padding": "8px" 
        }
    ).hide(axis="index") 
    display(styled_gen)
else:
    display(pd.DataFrame({"status": ["No generated answers"]}))

,category,bert_score_reference,bert_score_source,distinct_2,repetition_2,world_consistency
0,character description,0.7941,0.8038,0.9091,0.0618,0.9570
1,dialogue,0.8102,0.8090,0.9429,0.0381,0.9570
2,item description,0.8036,0.8090,0.8759,0.0602,0.9570
3,location description,0.8009,0.7988,0.8982,0.0796,0.9500
4,quest,0.8027,0.8117,0.8600,0.0775,0.9570
5,OVERALL,0.8023,0.8065,0.8972,0.0634,0.9556


category,generated_text
quest,"**Quest Title:** Echoes of Loyalty **Objective:** Uncover the truth behind a mysterious artifact and decide the fate of a minor faction caught in the crossfire of the Shattering. **Step 1: The Summons** In the heart of the Lands Between, you encounter a Tarnished named Eolande, who seeks your aid. She is a member of a small order dedicated to preserving ancient knowledge, now under threat from the chaos unleashed by the Shattering. **Step 2: The Artifact** Eolande presents an artifact she believes holds secrets that could change the course of the war among demigods. It's said to be a fragment of Marika's will, but its true purpose remains obscured. **Step 3: The Journey** Travel with Eolande to the ruins where the artifact was found. Along the way, you witness the aftermath of the Shattering—warring factions and creatures roaming the lands, each seeking power or change. **Step 4: The Revelation** Upon reaching the ruins, you discover that the artifact is not a piece of Marika's will but a communication device between her and an unknown entity. Intercepting a message, you learn that Marika had been secretly negotiating with the Fire Giants before the war, suggesting a betrayal of the Golden Order. **Step 5: The Choice** Confronted with this revelation, Eolande must decide whether to reveal this information to the Golden Order or protect it. Your choice will determine her loyalty: - **Loyalty to the Golden Order:** Hand over the artifact and the truth to Radagon, an aspect of the Greater Will. This could lead to a purge of traitors within the order. - **Defiance of the Golden Order:** Protect the secret and use it to forge an alliance with a faction against the Golden Order, potentially changing the tide of the war. **Step 6: The Consequence** Your choice will have far-reaching consequences. If you side with the Golden Order, you may uncover further truths about Marika's intentions, leading to a deeper understanding of her character and the true nature of the Shattering. Defying the order could lead to new alliances and a path towards a new world order. **Step 7: The Aftermath** Regardless of your choice, Eolande thanks you for revealing the truth she was destined to uncover. She offers you a piece of knowledge from her order's library as a token of gratitude—a secret that will aid you in your journey through the Lands Between."
dialogue,"**Spirit:** Wanderer of the Lands Between, I have awaited one such as you. The will of the world is in flux, and the Erdtree's power wanes with each passing moment. **Player:** Who are you? **Spirit:** I am but a whisper on the wind, a remnant of the Outer Gods' influence. I offer you a pact, Tarnished, to guide your steps through the shadows of our realm. **Player:** A pact? What does that entail? **Spirit:** In exchange for your loyalty and service, I shall reveal secrets long forgotten, paths untraveled, and the true nature of the power that once ruled these lands. The dragons, the Golden Order, even Queen Marika herself—all are threads in a tapestry woven by the Greater Will. **Player:** And what of the Greater Will? What role am I to play? **Spirit:** You are the hand that may stitch or tear asunder. As the Elden Ring shatters and reforms, so too does your destiny intertwine with the fate of the Lands Between. Accept this pact, and you shall walk a path illuminated by knowledge and power. **Player:** I accept. **Spirit:** So be it. The journey ahead is fraught with peril, but with newfound insight, you may unravel the mysteries that lie at the heart of our world."
character description,"**Character Profile: Eolande, the Wolf's Whisper** Eolande was once a member of the persecuted Omens, beings tied to the primordial crucible. As the Golden Order rose and began its relentless pursuit of those who defied The Greater Will, Eolande found herself on the run, her very existence a crime in the eyes of Queen Marika's order. In her flight, she stumbled upon a pack of 